# Load modules

In [22]:
import dask_jobqueue
import dask
import dask.distributed as dask_distributed
import os
import xarray as xr
import pyproj
import numpy as np
from gsw import f

import functions

# Open dask-cluster

In [4]:
%%time
cluster = dask_jobqueue.SLURMCluster(cores=4,memory='20GB',processes=1,queue='base', walltime='20:00:00',interface='ib0',local_directory='$TMPDIR')
client = dask.distributed.Client(cluster)
cluster.scale(jobs=6)
client

CPU times: user 309 ms, sys: 81.8 ms, total: 391 ms
Wall time: 3.01 s


/gxfs_home/geomar/smomw691/miniconda3/envs/py3_std/lib/python3.7/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 33349 instead
  f"Port {expected} is already in use.\n"


Connection method: Cluster object,Cluster type: dask_jobqueue.SLURMCluster
Dashboard: http://172.18.4.109:33349/status,
Dashboard: http://172.18.4.109:33349/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://172.18.4.109:36199,Workers: 0
Dashboard: http://172.18.4.109:33349/status,Total threads: 0
Started: Just now,Total memory: 0 B


# Define directories

In [16]:
hcs = 300 # horizontal chunk size
path_data = '/gxfs_work/geomar/smomw355/model_data/ocean-only/INALT60.L120-KRS0020/nemo/'
path_mask = path_data + 'suppl/2_INALT60.L120-KRS0020_mesh_mask.nc'
path_data += 'output/2_INALT60.L120-KRS0020_'

path_current = os.getcwd()
path_save = os.path.join(path_current, 'Fields_INALT60') 
os.makedirs(path_save, exist_ok=True)

path_RMS_w = !ls {path_current + 'RMS_w_model/'+'2013*'+'.nc'}

# Define period and region of interest

In [6]:
frq = '1d' #output frequency
yeara = 2013
day_select= 81 #168
day1 = 87
day2 = 191
depth=47

lon_c=15.7
lat_c=-37.8
pos_west,pos_east,pos_south,pos_north  = lon_c-2,lon_c+2,lat_c-2,lat_c+2

# Read data

In [8]:
#RMS profiles of vertical velocity
ds_RMS_model=xr.open_mfdataset(path_RMS_w)
RMS_w = ds_RMS_model.RMS_model.isel(time_counter=slice(day1,day2),z=slice(36,54)).mean(dim='z')

#INALT60
ds_mask = xr.open_dataset(path_mask,chunks={'x':hcs,'y':hcs})

paths = !ls {path_data + frq + '_' + str(yeara) + '*' + '_grid_U.nc'}
ds_u = xr.open_mfdataset(paths,chunks={'x':hcs,'y':hcs,'depthu':1,'time_counter':-1}).squeeze().rename({'depthu':'z'})

paths = !ls {path_data + frq + '_' + str(yeara) + '*' + '_grid_V.nc'}
ds_v = xr.open_mfdataset(paths,chunks={'x':hcs,'y':hcs,'depthv':1,'time_counter':-1}).squeeze().rename({'depthv':'z'})

paths = !ls {path_data + frq + '_' + str(yeara) + '*' + '_grid_T.nc'}
ds_ssh = xr.open_mfdataset(paths,chunks={'x':hcs,'y':hcs,'depthv':1,'time_counter':-1}).squeeze().rename({'deptht':'z'})

paths = !ls {path_data + frq + '_' + str(yeara) + '*' + '_grid_W.nc'}
ds_w = xr.open_mfdataset(paths,chunks={'x':hcs,'y':hcs,'depthw':1,'time_counter':-1}).squeeze().rename({'depthw':'z'})

ds_w_region = ds_w.where((ds_mask.nav_lat>pos_south-1.2) & (ds_mask.nav_lat<pos_north+1.2) & (ds_mask.nav_lon>pos_west-1.2) & (ds_mask.nav_lon<pos_east+1.2), drop=True)
ds_u_region = ds_u.where((ds_mask.nav_lat>pos_south-1.2) & (ds_mask.nav_lat<pos_north+1.2) & (ds_mask.nav_lon>pos_west-1.2) & (ds_mask.nav_lon<pos_east+1.2), drop=True)
ds_v_region = ds_v.where((ds_mask.nav_lat>pos_south-1.2) & (ds_mask.nav_lat<pos_north+1.2) & (ds_mask.nav_lon>pos_west-1.2) & (ds_mask.nav_lon<pos_east+1.2), drop=True)
ds_mask_region = ds_mask.where((ds_mask.nav_lat>pos_south-1.2) & (ds_mask.nav_lat<pos_north+1.2) & (ds_mask.nav_lon>pos_west-1.2) & (ds_mask.nav_lon<pos_east+1.2), drop=True)

/gxfs_home/geomar/smomw691/miniconda3/envs/py3_std/lib/python3.7/site-packages/xarray/core/indexing.py:1233: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  value = value[(slice(None),) * axis + (subkey,)]
/gxfs_home/geomar/smomw691/miniconda3/envs/py3_std/lib/python3.7/site-packages/xarray/core/indexing.py:1233: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': Tr

# Define a regular metric grid (needed for the eSQG comparison) for interpolation

In [11]:
dx=2 #regular spacing in km
lon_grid=ds_mask_region.isel(z=0).nav_lon
lat_grid=ds_mask_region.isel(z=0).nav_lat

#Define the grid
a = pyproj.Transformer.from_crs(4326,3395).transform(lon_grid,lat_grid)
y3 = a[0]
x3 = a[1]
x3min = np.nanmin(x3,1)
y3min = np.nanmin(y3,0)
Y3min,X3min = np.meshgrid(y3min, x3min)
x1=(x3-X3min)/1000
y1=(y3-Y3min)/1000
x_len = int(np.floor(np.amax(x1))+1)
y_len = int(np.floor(np.amax(y1))+1)
x_dim = np.linspace(0, x_len, int(x_len/dx), endpoint=False)
y_dim = np.linspace(0, y_len, int(y_len/dx), endpoint=False)
x2, y2 = np.meshgrid(x_dim, y_dim)

#Retrieve longitudes and latitudes back from the interpolation 
x2_proj = x2*1000 + x3min[0]
y2_proj = y2*1000 + y3min[0]
transformer_back = pyproj.Transformer.from_crs(3395, 4326, always_xy=True)
lat2, lon2 = transformer_back.transform(y2_proj,x2_proj)

#Make square
size_x=len(lon2[0,:])
size_y=len(lon2[:,0])

if size_y>size_x:
    delta_points=size_y-size_x
    lon2 = lon2[:size_x,:]
    lat2 = lat2[:size_x,:]
else:
    delta_points=size_x-size_y
    lon2 = lon2[:,:size_y]
    lat2 = lat2[:,:size_y]

# Compute surface strain and vorticity RMS time series

## Select data for the period and depth of interest

In [34]:
ds_v_surface = ds_v_region.isel(time_counter=slice(day1,day2), z=0)
ds_u_surface = ds_u_region.isel(time_counter=slice(day1,day2), z=0)

u_model = ds_u_surface.vozocrtx.where(ds_mask_region.isel(z=0).umask==1).squeeze()    
v_model = ds_v_surface.vomecrty.where(ds_mask_region.isel(z=0).vmask==1).squeeze() 

## Interpolate the fields on the horizontal on the regular 2-km grid

In [13]:
u_gridded_s = xr.apply_ufunc(interp_to_grid,
                     u_model.chunk(
        {"x": -1, "y": -1, "time_counter": 25}), x1, y1, x2, y2,
                     input_core_dims=[["y","x"], ["y","x"], ["y","x"], ["new_y","new_x"], ["new_y","new_x"]],
                     output_core_dims=[["new_y","new_x"]],
                     exclude_dims=set(("y","x")),
                     vectorize = True,
                     dask="parallelized",
                     output_dtypes=[u_model.dtype]).rename({"new_y": "y", "new_x": "x"})
    
v_gridded_s = xr.apply_ufunc(interp_to_grid,
             v_model.chunk(
{"x": -1, "y": -1, "time_counter": 25}), x1, y1, x2, y2,
             input_core_dims=[["y","x"], ["y","x"], ["y","x"], ["new_y","new_x"], ["new_y","new_x"]],
             output_core_dims=[["new_y","new_x"]],
             exclude_dims=set(("y","x")),
             vectorize = True,
             dask="parallelized",
             output_dtypes=[v_model.dtype]).rename({"new_y": "y", "new_x": "x"})

## Filter at 40 km

In [14]:
lfilter=40 
u_filtered = filtering_lanczos(u_gridded_s, lfilter)
v_filtered = filtering_lanczos(v_gridded_s, lfilter)

## Compute vorticity and strain

In [15]:
dudy = (u_filtered.shift(y=-1) - u_filtered.shift(y=1)) / (2*1000*dx)
dvdx = (v_filtered.shift(x=-1) - v_filtered.shift(x=1)) / (2*1000*dx)
dudx = (u_filtered.shift(x=-1) - u_filtered.shift(x=1)) / (2*1000*dx)
dvdy = (v_filtered.shift(y=-1) - v_filtered.shift(y=1)) / (2*1000*dx)

strain_model = ((dudx-dvdy)**2 + (dudy+dvdx)**2)**.5
strain_model = strain_model.isel(x=slice(50,-50),y=slice(50,-50)).compute()

zeta_model=dvdx-dudy
zeta_model = zeta_model.isel(x=slice(50,-50),y=slice(50,-50)).compute()

In [23]:
fcoriolis = abs(np.mean(f(lat2[50:-50,50:-50])))
RMS_strain = ((strain_model/fcoriolis)**2).mean(dim=['x', 'y'])**.5
RMS_vorticity = ((zeta_model/fcoriolis)**2).mean(dim=['x', 'y'])**.5

## Save data combined with RMS profiles of w

In [27]:
ds_to_save = xr.Dataset(
            data_vars=dict(
                RMS_strain=(["time_counter"], RMS_strain.values),
                RMS_w=(["time_counter"], RMS_w.isel(z=depth).values),
                RMS_zeta=(["time_counter"], RMS_vorticity.values)
                
            ),
            coords=dict(
                time_counter=(["time_counter"], RMS_w.time_counter.values)
            ) 
        )
ds_to_save.to_netcdf(path=path_save+'RMS_timeseries_INALT60.nc')

# Compute vorticity, strain and w maps for a given day

## Interpolate w on the vertical at SSH grid points since the eSQG will produce w estimates at SSH grid points

In [29]:
w_model = ds_w_region.isel(time_counter=day1+day_select)
mask_1000 = ds_ssh.z <= 1000
ds_mask_region = ds_mask_region.where(mask_1000, drop=True)

w_org = w_model.vovecrtz  
z1=w_model.z
z2=ds_ssh.z.where(mask_1000, drop=True).rename({"z": "z_new"})

w_gridded_vert = xr.apply_ufunc(interp_to_prof,
                     w_org.chunk(
        {"x": 100, "y": 100, "z":-1}), z1, z2,
                     input_core_dims=[["z"], ["z"], ["z_new"]],
                     output_core_dims=[["z_new"]],
                     exclude_dims=set(("z")),
                     vectorize = True,
                     dask="parallelized",
                     output_dtypes=[w_org.dtype]).rename({"z_new": "z"})

w_gridded_vert=w_gridded_vert.where(ds_mask_region.tmask==1).isel(z=depth).squeeze()*60*60*24

# Interpolate the fields on the horizontal on the regular 2-km grid

In [33]:
w_gridded = xr.apply_ufunc(interp_to_grid,
                     w_gridded_vert.chunk(
        {"x": -1, "y": -1}), x1, y1, x2, y2,
                     input_core_dims=[["y","x"], ["y","x"], ["y","x"], ["new_y","new_x"], ["new_y","new_x"]],
                     output_core_dims=[["new_y","new_x"]],
                     exclude_dims=set(("y","x")),
                     vectorize = True,
                     dask="parallelized",
                     output_dtypes=[w_gridded_vert.dtype]).rename({"new_y": "y", "new_x": "x"})

## Filter at 40 km

In [40]:
lfilter=40
w_filtered = filtering_lanczos(w_gridded, lfilter).compute()

## Vorticity and strain

In [47]:
strain_model = ((dudx-dvdy)**2 + (dudy+dvdx)**2)**.5
strain_model_select = strain_model.isel(time_counter=day_select).compute()

zeta_model=dvdx-dudy
zeta_model_select = zeta_model.isel(time_counter=day_select).compute()

## Save the vorticity, strain and w fields

In [48]:
ds_to_save = xr.Dataset(
        data_vars=dict(
            zeta=(["y", "x"], zeta_model_select.values),
            w=(["y", "x"], w_filtered.values),
            strain=(["y", "x"], strain_model_select.values),
            latitude=(["y", "x"], lat2),
            longitude=(["y", "x"], lon2)
        ),
        coords=dict(
            y=("y", np.arange(len(lat2[:,0]))),
            x=("x", np.arange(len(lat2[0,:])))
        ) 
    )

ds_to_save.to_netcdf(path=path_save+'INALT60_zeta_w_strain.nc')

# Extract SSH field over a larger area

In [51]:
dx=2
lon_grid=ds_mask.nav_lon
lat_grid=ds_mask.nav_lat

a = pyproj.Transformer.from_crs(4326,3395).transform(lon_grid,lat_grid) # project WGS84 onto metric grid
y3 = a[0]
x3 = a[1]
x3min = np.nanmin(x3,1)
y3min = np.nanmin(y3,0)
Y3min,X3min = np.meshgrid(y3min, x3min)
x1=(x3-X3min)/1000
y1=(y3-Y3min)/1000
x_len = int(np.floor(np.amax(x1))+1)
y_len = int(np.floor(np.amax(y1))+1)
x_dim = np.linspace(0, x_len, int(x_len/dx), endpoint=False)
y_dim = np.linspace(0, y_len, int(y_len/dx), endpoint=False)
x2, y2 = np.meshgrid(x_dim, y_dim)

x2_proj = x2*1000 + x3min[0]
y2_proj = y2*1000 + y3min[0]
transformer_back = pyproj.Transformer.from_crs(3395, 4326, always_xy=True)
lat_ssh, lon_shh = transformer_back.transform(y2_proj,x2_proj)

ds_ssh_select = ds_ssh.isel(time_counter=day_select+day1)
ssh_model = ds_ssh_select.sossheig.where(ds_mask.tmask.isel(z=0)==1).squeeze()  

ssh_gridded = xr.apply_ufunc(interp_to_grid,
             ssh_model.chunk(
{"x": -1, "y": -1}), x1, y1, x2, y2,
             input_core_dims=[["y","x"], ["y","x"], ["y","x"], ["new_y","new_x"], ["new_y","new_x"]],
             output_core_dims=[["new_y","new_x"]],
             exclude_dims=set(("y","x")),
             vectorize = True,
             dask="parallelized",
             output_dtypes=[ssh_model.dtype]).rename({"new_y": "y", "new_x": "x"})

ssh_filtered = xr.apply_ufunc(
    filter_convolution2d,
    ssh_gridded,lfilter,
    input_core_dims=[["y","x"],[]],
    output_core_dims=[["y","x"]],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[ssh_gridded.dtype]).compute()

## Save SSH field

In [52]:
ds_to_save = xr.Dataset(
        data_vars=dict(
            ssh=(["y", "x"], ssh_filtered.values),
            latitude=(["y", "x"], lat_ssh),
            longitude=(["y", "x"], lon_shh)
        ),
        coords=dict(
            y=("y", np.arange(lat_ssh.shape[0])),
            x=("x", np.arange(lat_ssh.shape[1]))
        ) 
    )

ds_to_save.to_netcdf(path=path_save+'INALT60_ssh.nc')